# Replogle

The Replogle data was obtained from `pertpy`:
- K562: `pertpy.data.replogle_2022_k562_essential()`
- RPE1: `pertpy.data.replogle_2022_rpe1()`
- K562 gwps: `pertpy.data.replogle_2022_k562_gwps()`

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

In [ ]:
adata_ess_k562 = sc.read_h5ad("../data/replogle_2022_k562_essential.h5ad")
adata_ess_rpe1 = sc.read_h5ad("../data/replogle_2022_rpe1.h5ad")

In [ ]:
adata_ess_k562

AnnData object with n_obs × n_vars = 310385 × 8563
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'

In [ ]:
adata_ess_rpe1

AnnData object with n_obs × n_vars = 247914 × 8749
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo', 'celltype'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'

In [ ]:
adata_ess_k562.obs['gene'].nunique()

2058

In [ ]:
n_perts = adata_ess_k562.obs['perturbation'].nunique()
print(f'n perts in k562: {n_perts}')
n_perts = adata_ess_rpe1.obs['perturbation'].nunique()
print(f'n perts in rpe1: {n_perts}')

n perts in k562: 2058
n perts in rpe1: 2394


## Perturbation filtering
Follow the preprocessing steps in https://github.com/yhr91/GEARS_misc/blob/main/data/preprocessing/Replogle_2022_preprocess.ipynb to keep only strong perturbations.
Metadata obtained from TableS2 of the original publication: https://www.sciencedirect.com/science/article/pii/S0092867422005979?via%3Dihub

In [ ]:
supp2_ess_k562 = pd.read_excel('../data/1-s2.0-S0092867422005979-mmc2.xlsx',
                     sheet_name='TabB_K562_day6_summary_stat')
supp2_ess_rpe1 = pd.read_excel('../data/1-s2.0-S0092867422005979-mmc2.xlsx',
                     sheet_name='TabC_RPE1_summary_statistic')

In [ ]:
def get_strong_perts(supp):
    filtered = supp[supp['Number of DEGs (anderson-darling)']>50]
    filtered = filtered[filtered['percent knockdown']<=-0.3]
    filtered = filtered[filtered['number of cells (filtered)']>25]
    strong_perts = filtered['genetic perturbation'].values
    strong_perts = [s.split('_')[1] for s in strong_perts]
    return strong_perts

strong_perts_ess_k562 = get_strong_perts(supp2_ess_k562)
strong_perts_ess_rpe1 = get_strong_perts(supp2_ess_rpe1)
len(strong_perts_ess_k562), len(strong_perts_ess_rpe1)

(1092, 1564)

In [ ]:
strong_perts_ess_k562 = strong_perts_ess_k562 + ['non-targeting']
strong_perts_ess_rpe1 = strong_perts_ess_rpe1 + ['non-targeting']

In [ ]:
pert_filter_k562 = adata_ess_k562[adata_ess_k562.obs['gene'].isin(strong_perts_ess_k562)]
pert_filter_rpe1 = adata_ess_rpe1[adata_ess_rpe1.obs['gene'].isin(strong_perts_ess_rpe1)]

In [ ]:
pert_filter_k562

View of AnnData object with n_obs × n_vars = 192648 × 8563
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'

In [ ]:
pert_filter_rpe1

View of AnnData object with n_obs × n_vars = 175398 × 8749
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo', 'celltype'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'

In [ ]:
n_perts = pert_filter_k562.obs['perturbation'].nunique()
print(f'n perts in k562: {n_perts}')
n_perts = pert_filter_rpe1.obs['perturbation'].nunique()
print(f'n perts in rpe1: {n_perts}')

n perts in k562: 1093
n perts in rpe1: 1544


Remove cells that don't show knock down effect

In [ ]:
def filter_cells_by_pert_effect(adata, k=10):
    
    perc_underk = []
    subset_idxs = []
    ctrl_adata = adata[adata.obs['gene'] == 'non-targeting']
    
    for g in adata.obs['gene'].unique():        
        
        subset = adata[adata.obs['gene'] == g]
        
        if g == 'non-targeting':
            subset_idxs.append(subset.obs.index.values)
            continue
        
        try:
            gene_loc = np.where(adata.var_names == g)[0][0]
            thresh = np.percentile(ctrl_adata.X[:,gene_loc],k)
            perc_underk.append(sum(subset.X[:,gene_loc]>thresh))

            subset_idxs.append(subset.obs.index[subset.X[:,gene_loc]<=thresh].values)
        except:
            subset_idxs.append(subset.obs.index.values)
            
    subset_idxs = strong_perts = [item for sublist in subset_idxs for item in sublist]
    filtered_adata = adata[subset_idxs,:]
            
    return perc_underk, filtered_adata

In [ ]:
perc_underk_ess_k562, filtered_adata_ess_k562 = filter_cells_by_pert_effect(pert_filter_k562)
perc_underk_ess_rpe1, filtered_adata_ess_rpe1 = filter_cells_by_pert_effect(pert_filter_rpe1)

In [ ]:
filtered_adata_ess_k562

View of AnnData object with n_obs × n_vars = 162751 × 8563
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'

In [ ]:
filtered_adata_ess_rpe1

View of AnnData object with n_obs × n_vars = 162734 × 8749
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo', 'celltype'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'

In [ ]:
filtered_adata_ess_k562.write_h5ad('../data/replogle_2022_k562_essential_filtered.h5ad')
filtered_adata_ess_rpe1.write_h5ad('../data/replogle_2022_rpe1_filtered.h5ad')

In [ ]:
adata = sc.read_h5ad("../data/replogle_2022_k562_gwps.h5ad")

In [ ]:
adata

AnnData object with n_obs × n_vars = 1989578 × 8248
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'

In [ ]:
supp2 = pd.read_excel('../data/1-s2.0-S0092867422005979-mmc2.xlsx',
                     sheet_name='TabA_K562_day8_summary_stat')

In [ ]:
def get_strong_perts(supp):
    filtered = supp[supp['Number of DEGs (anderson-darling)']>50]
    filtered = filtered[filtered['percent knockdown']<=-0.3]
    filtered = filtered[filtered['number of cells (filtered)']>25]
    strong_perts = filtered['genetic perturbation'].values
    strong_perts = [s.split('_')[1] for s in strong_perts]
    return strong_perts

strong_perts = get_strong_perts(supp2)
len(strong_perts)

1931

In [ ]:
strong_perts= strong_perts + ['non-targeting']

In [ ]:
pert_filter = adata[adata.obs['gene'].isin(strong_perts)]

In [ ]:
pert_filter

View of AnnData object with n_obs × n_vars = 444602 × 8248
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'

In [ ]:
n_perts = pert_filter.obs['perturbation'].nunique()
print(f'n perts: {n_perts}')

n perts: 1929


Remove cells that don't show knock down effect

In [ ]:
perc_underk, filtered_adata= filter_cells_by_pert_effect(pert_filter)

In [ ]:
filtered_adata

View of AnnData object with n_obs × n_vars = 394089 × 8248
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'

In [ ]:
filtered_adata.write_h5ad('../data/replogle_2022_k562_gwps_filtered.h5ad')

### preprocessing

K562

In [ ]:
%%bash
adata_path=../data/replogle_2022_k562_essential_filtered.h5ad
cell_type=K562
dataset_name=replogle_k562

python ../scripts/preprocess_data.py ${adata_path} ${cell_type} ${dataset_name}

RPE1

In [ ]:
%%bash
adata_path=../data/replogle_2022_rpe1_filtered.h5ad
cell_type=RPE1
dataset_name=replogle_rpe1

python ../scripts/preprocess_data.py ${adata_path} ${cell_type} ${dataset_name}

K562 gwps

In [ ]:
%%bash
adata_path=../data/replogle_2022_k562_gwps_filtered.h5ad
cell_type=K562
dataset_name=replogle_k562_gwps

python ../scripts/preprocess_data.py ${adata_path} ${cell_type} ${dataset_name}

# Nadig

The pre-filtered data was directly download from https://huggingface.co/datasets/arcinstitute/Replogle-Nadig-Preprint/tree/main

In [ ]:
%%bash
adata_path=../data/GSE264667_jurkat_raw_singlecell_01_filtered.h5ad
cell_type=Jurkat
dataset_name=nadig_jurkat

python ../scripts/preprocess_data.py ${adata_path} ${cell_type} ${dataset_name}

In [ ]:
%%bash
adata_path=../data/GSE264667_hepg2_raw_singlecell_01_filtered.h5ad
cell_type=hepg2
dataset_name=nadig_hepg2

python ../scripts/preprocess_data.py ${adata_path} ${cell_type} ${dataset_name}

# Norman

Norman dataset was download via `pertpy.data.norman_2019()`. 
We follow GEARS tutorial for preprocessing (https://github.com/yhr91/GEARS_misc/blob/f88211870dfa89c38a2eedbd69ca1abd28a25f3c/data/preprocessing/Norman19.ipynb).

In [ ]:
adata = sc.read_h5ad("../data/norman_2019.h5ad")

In [ ]:
adata = adata[adata.obs['guide_identity'] != "NegCtrl1_NegCtrl0__NegCtrl1_NegCtrl0"]

In [ ]:
adata.obs['condition'] = adata.obs['guide_identity']

import re
for i in np.unique(adata.obs["condition"]):
   m = re.match(r"NegCtrl(.*)_NegCtrl(.*)+NegCtrl(.*)_NegCtrl(.*)", i)
   if m :
        adata.obs["condition"].replace(i,"ctrl",inplace=True)

In [ ]:
old_pool = []
for i in np.unique(adata.obs["condition"]):
    if i == "ctrl":
        old_pool.append(i)
        continue
    split = i.split("__")[1]
    split = split.split("_")
    for j, string in enumerate(split):
        if "NegCtrl" in split[j]:
            split[j] = "ctrl"
    if len(split) == 1:
        if split[0] in old_pool:
            print("old:",i, "new:",split[0])
        adata.obs["condition"].replace(i,split[0],inplace=True)
        old_pool.append(split[0])
    else:
        if f"{split[0]}+{split[1]}" in old_pool:
            print("old:",i, "new:",f"{split[0]}+{split[1]}")
        adata.obs["condition"].replace(i, f"{split[0]}+{split[1]}",inplace=True)
        old_pool.append(f"{split[0]}+{split[1]}")

old: HOXC13_NegCtrl0__HOXC13_NegCtrl0_2 new: HOXC13+ctrl
old: TGFBR2_IGDCC3__TGFBR2_IGDCC3_2 new: TGFBR2+IGDCC3
old: ZBTB10_NegCtrl0__ZBTB10_NegCtrl0_2 new: ZBTB10+ctrl


In [ ]:
adata.write_h5ad('../data/norman_2019_filtered.h5ad')

In [ ]:
%%bash
adata_path=../data/norman_2019_filtered.h5ad
cell_type=K562
dataset_name=norman_combo

python ../scripts/preprocess_data.py ${adata_path} ${cell_type} ${dataset_name}

# Make data splits

In [1]:
import numpy as np
import scanpy as sc

In [2]:
def make_split(adata, train_fraction=0.75, val_fraction=0.1, test_fraction=0.15, random_seed=42, save_path=None):
    # Get all unique conditions
    all_conditions = adata.obs['condition'].unique().tolist()
    non_ctrl = [c for c in all_conditions if c != 'ctrl']

    # Shuffle for reproducibility
    rng = np.random.default_rng(seed=random_seed)
    shuffled = rng.permutation(non_ctrl).tolist()

    # Split sizes (based on non-ctrl conditions)
    n = len(shuffled)
    n_train = int(np.round(n * train_fraction))
    n_val = int(np.round(n * val_fraction))
    n_test = n - n_train - n_val

    train_conds = shuffled[:n_train]
    val_conds = shuffled[n_train:n_train + n_val]
    test_conds = shuffled[n_train + n_val:]

    # 'ctrl' always goes to train
    train_conds = train_conds + ['ctrl']

    split_dict = {
        'train': train_conds,
        'val': val_conds,
        'test': test_conds,
    }

    print(f"Total conditions: {len(all_conditions)}  (ctrl + {len(non_ctrl)} perturbations)")
    print(f"Train: {len(train_conds)} | Val: {len(val_conds)} | Test: {len(test_conds)}")
    
    # Save if path provided
    if save_path:
        import pickle
        import os
        # Create directory if it doesn't exist
        save_dir = os.path.dirname(save_path)
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
        with open(save_path, 'wb') as f:
            pickle.dump(split_dict, f)
        print(f"Split saved to: {save_path}")
    
    return split_dict

In [ ]:
datasets = ['replogle_k562', 'replogle_rpe1', 'nadig_jurkat', 'nadig_hepg2']
for dataset in datasets:
    adata = sc.read_h5ad(f'../data/processed_{dataset}.h5ad', backed='r')
    for i, seed in enumerate([42, 103, 32]):
        make_split(adata, random_seed=seed, save_path=f"../data/gears/{dataset}/splits/{dataset}_single_{i+1}_0.75.pkl")

In [ ]:
from gears import PertData
import os

pert_data_dir = "../data/gears/norman_combo"
pert_data = PertData(data_path=os.path.dirname(pert_data_dir))

pert_data.load(data_path = pert_data_dir)

for seed in [1, 2, 3]:
    pert_data.prepare_split(split='simulation', seed = seed)